<figure>
    <center> <img src="../../images/mai_banner.png"  style="width:1000px;height:180px;" ></center>
</figure>

# Lab 10: Linear Regression con Scikit Learn

## 1. Instalación de dependencias

Instamos las dependecias necesarias para el laboratorio en modo silencioso

In [ ]:
%pip install -q numpy==2.2.0
%pip install -q pandas==2.2.3
%pip install -q scikit-learn==1.6.0
%pip install -q matplotlib==3.9.3

El siguiente paso sera importar las dependencias

In [ ]:
# Esta linea de codigo importa la biblioteca numpy
import numpy as np
# Esta linea de codigo importa la biblioteca matplotlib para la visualización de gráficos
import matplotlib.pyplot as plt
# Esta linea de codigo importa la biblioteca pandas para el manejo de datos en forma de tablas
import pandas as pd
# Esta linea de codigo habilita la visualización de gráficos dentro del notebook
%matplotlib inline

## 2 Cargando el dataset

En esta línea de codigo cargamos la url donde se encuentra el dataset que utilizaremos en el laboratorio.

In [ ]:
url= "https://raw.githubusercontent.com/rivalcoba/ml-2026b-virtual/075c5ab3c334efa216e9e8bd71ecc21950d62278/labs/lab10-LR_scikitlearn/dataset/FuelConsumptionCo2.csv"

El nombre del dataset es FuelConsumption este dataset contiene la calificaciones del consumo de combustible y la estimación de las emisiones de CO2 de diversas marcas de vehículos puestas a la venta en Canada.

Este es un ejemplo de uno de los registros del dataset.
- **MODEL YEAR** e.g. 2014
- **MAKE** e.g. VOLVO
- **MODEL** e.g. S60 AWD
- **VEHICLE CLASS** e.g. COMPACT
- **ENGINE SIZE** e.g. 3.0
- **CYLINDERS** e.g 6
- **TRANSMISSION** e.g. AS6
- **FUEL TYPE** e.g. Z
- **FUEL CONSUMPTION in CITY(L/100 km)** e.g. 13.2
- **FUEL CONSUMPTION in HWY (L/100 km)** e.g. 9.5
- **FUEL CONSUMPTION COMBINED (L/100 km)** e.g. 11.5
- **FUEL CONSUMPTION COMBINED MPG (MPG)** e.g. 25
- **CO2 EMISSIONS (g/km)** e.g. 182

Tipo de combustible
- X. Gasolina regular (Magna o verde en México, de 87 octanos).
- Z. Gasolina Premium (roja en México, de 91+ octanos).
- D. Diésel.
- E. Etanol (E85).
- B. Electricidad (vehículos 100% eléctricos).
- N. Gas natural.

A continuación, vamos a crear un modelo de regresión lineal múltiple para predecir las emisiones de CO2 de vehículos nuevos basándonos en descriptores seleccionados.

Para cargar los datos usaremos la función de pandas llamada pd.read_csv() y seguido de eso imprimiremos una muestra del dataset usando el método head().

In [ ]:
# Cargando el dataset, en este caso df significa DataFrame
df = pd.read_csv(url)
# Mostrando una muestra aleatoria de 5 registros del dataset
df.sample(5)

## 3 Explorando y seleccionando descriptores

Para extraer algunas características de cada descriptor usaremos el comando describe() del objeto dataframe de la siguiente manera:

```python
df.describe()
```

In [ ]:
df.describe()

A continuación te explico que significa cada descriptor:

- `count`: número de valores no nulos en la columna.
- `mean`: promedio de los valores de la columna.
- `std`: desviación estándar de los valores de la columna.
- `min`: valor mínimo de la columna.
- `25%`: primer cuartil (25% de los datos son menores o iguales a este valor).
- `50%`: mediana o segundo cuartil (50% de los datos son menores o iguales a este valor).
- `75%`: tercer cuartil (75% de los datos son menores o iguales a este valor).
- `max`: valor máximo de la columna.

> ❓ ¿Que descriptores no aparecen?

<details>
<summary><font size='5', color='red'><b>Respuesta</b></font></summary>

Al observar el resultado de `df.describe()`, notamos que no aparecen los descriptores para las columnas no numéricas, como las categóricas o de tipo objeto.

Si observamos bien la tabla algunos descriptores no aparecerán, por ejemplo: transmisión, tipo de motor, esto debido a que no son variables numéricas. Otra observación interesante es que el año del modelo es el mismo para todos, por lo que no es información que aporte a la clasificación. 
</details>

A continiación eliminaremos del dataset las columnas que no son numéricas, ya que no aportan información relevante para el análisis que realizaremos. Estas corresponden a:
- `TRANSMISSION` ese se elimina porque no es una variable numérica.
- `FUELTYPE` ese se elimina porque no es una variable numérica.
- `MODELYEAR` ese se elimina porque no aporta información relevante para la clasificación.
- `MAKE` ese se elimina porque no es una variable numérica.
- `MODEL` ese se elimina porque no es una variable numérica.
- `VEHICLECLASS` ese se elimina porque no es una variable numérica.

In [ ]:
# Eliminando columnas categóricas y cualquier columnas sin utilidad
# axis indica si se eliminan filas (0) o columnas (1)
df = df.drop(['MODELYEAR', 'MAKE', 'MODEL', 'VEHICLECLASS', 'TRANSMISSION', 'FUELTYPE',],axis=1)

## 4 Matriz de correlación

Ahora analizaremos la matriz de correlación, que muestra las relaciones entre pares de descriptores e indica su nivel de independencia. Esta matriz nos permitirá identificar qué descriptores pueden aportar información más relevante para nuestro modelo de predicción.

> 💡 Debes eliminar cualquier dependencia o correlación fuerte *entre características* seleccionando la mejor de cada grupo correlacionado.

In [ ]:
# Obteniendo la matriz de correlación
df.corr()

Veamos las siguientes observaciones:
- **Todos los descriptores exhiben una alta correlación**, superando el 85% de magnitud. Esto nos permite concluir que estos descriptores son buenos candidatos para nuestro modelo de regresión lineal.
- Podemos observar que **los cilindros y el tamaño del motor están fuertemente correlacionados**, por lo que debemos quedarnos con tan solo uno de ellos. Elegiremos tamaño de motor ya que está más correlacionado con la variable objetivo.
- De manera similar, cada una de las cuatro variables de economía de combustible está altamente correlacionada entre sí.
    - Como `FUELCONSUMPTION_COMB_MPG` es la que tiene mayor correlación con la variable objetivo, puedes eliminar las otras: `FUELCONSUMPTION_CITY`, `FUELCONSUMPTION_HWY`, `FUELCONSUMPTION_COMB`.
- Observa que `FUELCONSUMPTION_COMB` y `FUELCONSUMPTION_COMB_MPG` no están perfectamente correlacionadas. Sin embargo, deberían estarlo, ya que miden la misma propiedad en diferentes unidades. En la práctica, investigarías por qué ocurre esto. Podrías descubrir que algunos o todos los datos no son utilizables tal como están.

Derivado de lo anterior eliminaremos los siguientes descriptores:
- `CYLINDERS` porque está fuertemente correlacionado con el tamaño del motor.
- `FUELCONSUMPTION_CITY` porque está altamente correlacionada con `FUELCONSUMPTION_COMB_MPG`.
- `FUELCONSUMPTION_HWY` porque está altamente correlacionada con `FUELCONSUMPTION_COMB_MPG`.
- `FUELCONSUMPTION_COMB` porque está altamente correlacionada con `FUELCONSUMPTION_COMB_MPG`.

In [ ]:
# Eliminando descriptores debiles
df = df.drop(['CYLINDERS', 'FUELCONSUMPTION_CITY', 'FUELCONSUMPTION_HWY','FUELCONSUMPTION_COMB',],axis=1)

Imprimiremos las primeras 9 filas

In [ ]:
# Mostrando las primeras 9 filas del DataFrame después de eliminar descriptores débiles
df.head(9)

Podemos observar que serán tan solo dos descriptores los que nos permitirán entrenar al modelo.

Podemos auxiliarnos de una visualización para confirmar que hemos seleccionado los mejores descriptores y que no son redundantes entre sí.

A continuación, crearemos una matriz de dispersión que mostrará gráficas para cada par de descriptores. La diagonal de esta matriz presentará el histograma individual de cada descriptor.pairplot(df)
```

In [ ]:
# Creando una matriz de dispersión para visualizar los descriptores seleccionados
axes = pd.plotting.scatter_matrix(df, figsize=(8, 8), alpha=0.2)

# Rotate axis labels to improve readability
for ax in axes.flatten():
    ax.xaxis.label.set_rotation(90)  # Rotate x-axis labels vertically
    ax.yaxis.label.set_rotation(0)   # Keep y-axis labels horizontal
    ax.yaxis.label.set_ha('right')   # Align y-axis labels to the right

# Adjust the layout to make sure everything fits without overlap
plt.tight_layout()

# Remove space between subplots for a more compact visualization
plt.gcf().subplots_adjust(wspace=0, hspace=0)

# Display the scatter matrix
plt.show()

> 💡 Como puedes ver, la relación entre `FUELCONSUMPTION_COMB_MPG` y `CO2EMISSIONS` no es lineal. Esto sugiere que un modelo de regresión lineal simple podría no ser el más adecuado para capturar esta relación.

# 5. Extrayendo descriptores

Es momento de extraer las columnas del dataset para colocarlas en arreglos de Numpy, para ello ejecutamos el siguiente código:

In [ ]:
# Extrayendo descriptores y variable objetivo
# X contiene los descriptores seleccionados (columnas 0 y 1)
X = df.iloc[:,[0,1]].to_numpy()

# y contiene la variable objetivo (columna 2)
y = df.iloc[:,[2]].to_numpy()

> ⚠️ La función iloc en pandas es un método de indexación basado en posición que permite acceder a filas y columnas de un DataFrame utilizando índices enteros (números de fila y columna).
> 
> En este caso, `df.iloc[:,[0,1]]` selecciona todas las filas (`:`) y las columnas 0 y 1, mientras que `df.iloc[:,[2]]` selecciona todas las filas y la columna 2.
> 
> Esto es útil para preparar los datos antes de entrenar un modelo de regresión lineal, ya que `X` contendrá los descriptores y `y` la variable objetivo.

# 6. Proceso de Datos

Un paso importante es estandarizar los descriptores de entrada. Con esto evitaremos que el modelo favorezca a un descriptor simplemente por tener valores de mayor magnitud.

In [ ]:
# Importar el módulo preprocessing de scikit-learn que contiene herramientas 
# para transformar datos
from sklearn import preprocessing

# Crear un objeto StandardScaler que estandarizará los datos 
# (media=0, desviación estándar=1)
std_scaler = preprocessing.StandardScaler()

# Aplicar la estandarización a los datos X:
# 1. Calcula la media y desviación estándar con fit()
# 2. Transforma los datos restando la media y dividiendo por la desviación estándar
X_std = std_scaler.fit_transform(X)

En la práctica, si quieres evaluar correctamente tu modelo, definitivamente no deberías aplicar tales operaciones al conjunto de datos completo, sino a los datos de entrenamiento y prueba por separado.

Con el siguiente código evaluaremos el dataset procesado utilizando la estandarización aplicada previamente.

In [ ]:
# Descripción estadística del dataset estandarizado
pd.DataFrame(X_std).describe().round(2)

Podemos observar que la media es 0 y la desviación estándar es 1, lo cual caracteriza a una variable normalizada.

# 7. Creación de datasets de prueba y entrenamiento

Vamos a dividir el conjunto de datos en 2 partes de manera aleatorea, un 80% de los datos se iran a entrenamiento y otro 20% se ira a prueba.

In [ ]:
from sklearn.model_selection import train_test_split
# División del dataset en entrenamiento y prueba
# random_state se utiliza para asegurar la reproducibilidad de la división del dataset.
X_train, X_test, y_train, y_test = train_test_split(X_std,y,
                                                    test_size=0.2,
                                                    random_state=42)

# 8. Construcción del regresor lineal
TODO: Construir un modelo de regresión lineal utilizando los datos de entrenamiento.